# 02 - Explore railway station records

**Source:** `emplacement-des-gares-idf.csv` (full file).
**Module:** `scripts/datasets/stations.py`.

Understand station names, IDs, line IDs, modes and coordinate fields. The source is by
station and line; several rows with one station ID can be legitimate. No line reference
or workplace data is joined.


In [65]:
import sys
from pathlib import Path
import pandas as pd
import plotly.io as pio
from IPython.display import display

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if not (ROOT / 'pyproject.toml').is_file():
    raise FileNotFoundError('Open the hackathon project folder before running this notebook.')
if not Path(sys.executable).resolve().is_relative_to(ROOT / '.venv'):
    raise RuntimeError('Select the project .venv kernel, then restart and run all.')
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import importlib
from scripts.datasets import stations

# Refresh the module if stations.py changed while this kernel was running.
importlib.reload(stations)
from scripts.exploration import column_summary, repeated_keys, save_parquet

pio.renderers.default = 'plotly_mimetype'
print('Python:', sys.executable)


Python: d:\Hackathon Mobilités 2026 Spécial Dataviz\.venv\Scripts\python.exe


In [66]:
source = ROOT / stations.FILENAME
raw = stations.load(source)


In [67]:
print('Loaded rows and columns:', raw.shape)
display(raw.head())
raw.info()


Loaded rows and columns: (1287, 29)


,Geo Point,Geo Shape,gares_id,id_gare_g,nom_long,nom_so_gar,nom_su_gar,id_ref_ZdC,nom_ZdC,id_ref_ZdA,...,tertram,terval,exploitant,idf,principal,date_mise_,x,y,picto ligne,nom_iv
0,"48.89767181948505, 2.3298303344940856","{""coordinates"": [2.329830334494086, 48.8976718...",1211,114071,Porte de Saint-Ouen,,,72078,Porte de Saint-Ouen,473689,...,0,0,RATP,1,0,2018-11-24T01:00:00+01:00,650866.4949,6866615.8591,https://data.iledefrance-mobilites.fr/api/expl...,Porte de Saint-Ouen
1,"48.90101049575492, 2.2135566086401575","{""coordinates"": [2.213556608640157, 48.9010104...",578,388,Nanterre-Université,,,70956,Nanterre Université,43170,...,0,0,RATP,1,1,1972-10-01T01:00:00+01:00,642345.8425,6867065.7254,https://data.iledefrance-mobilites.fr/api/expl...,Nanterre-Université
2,"48.970665048322914, 2.0777317645655926","{""coordinates"": [2.077731764565593, 48.9706650...",6,127,Achères-Ville,,,73604,Achères Ville,46647,...,0,0,SNCF,1,0,1988-05-29T02:00:00+02:00,632479.4115,6874918.7325,https://data.iledefrance-mobilites.fr/api/expl...,Achères-Ville
3,"48.88521121820875, 2.1559034718965178","{""coordinates"": [2.155903471896518, 48.8852112...",172,98,Châtou-Croissy,,,64483,Châtou - Croissy,53783,...,0,0,RATP,1,0,1972-10-01T01:00:00+01:00,638100.6669,6865352.6049,https://data.iledefrance-mobilites.fr/api/expl...,Châtou-Croissy
4,"48.83921929995196, 2.633210913417631","{""coordinates"": [2.633210913417631, 48.8392192...",473,6,Lognes,,,68123,Lognes,43152,...,0,0,RATP,1,0,1980-12-19T01:00:00+01:00,673078.4315,6859970.1835,https://data.iledefrance-mobilites.fr/api/expl...,Lognes


<class 'pandas.DataFrame'>
RangeIndex: 1287 entries, 0 to 1286
Data columns (total 29 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   Geo Point    1287 non-null   string
 1   Geo Shape    1287 non-null   string
 2   gares_id     1287 non-null   string
 3   id_gare_g    1287 non-null   string
 4   nom_long     1287 non-null   string
 5   nom_so_gar   1287 non-null   string
 6   nom_su_gar   1287 non-null   string
 7   id_ref_ZdC   1287 non-null   string
 8   nom_ZdC      1287 non-null   string
 9   id_ref_ZdA   1287 non-null   string
 10  nom_ZdA      1287 non-null   string
 11  idrefliga    1287 non-null   string
 12  idrefligc    1287 non-null   string
 13  res_com      1287 non-null   string
 14  indice_lig   1287 non-null   string
 15  mode         1287 non-null   string
 16  tertrain     1287 non-null   string
 17  terrer       1287 non-null   string
 18  termetro     1287 non-null   string
 19  tertram      1287 non-null   string
 2

## Clean a copy and inspect the columns

The original file and `raw` table stay unchanged. Strip surrounding spaces and turn
blank cells into missing values; preserve special codes such as `NA` and `ZZ`.
Dataset-specific numeric conversions are documented in the module's `clean()` function.
Unexpected numeric text raises an error for inspection. No rows are dropped or deduplicated.


In [68]:
cleaned = stations.clean(raw)
assert len(cleaned) == len(raw)
display(cleaned.head())
display(column_summary(cleaned))


,Geo Point,Geo Shape,gares_id,id_gare_g,nom_long,nom_so_gar,nom_su_gar,id_ref_ZdC,nom_ZdC,id_ref_ZdA,...,idf,principal,date_mise_,x,y,picto ligne,nom_iv,latitude,longitude,coordinates_in_bounds
0,"48.89767181948505, 2.3298303344940856","{""coordinates"": [2.329830334494086, 48.8976718...",1211,114071,Porte de Saint-Ouen,<NA>,<NA>,72078,Porte de Saint-Ouen,473689,...,1,0,2018-11-24T01:00:00+01:00,650866.4949,6866615.8591,https://data.iledefrance-mobilites.fr/api/expl...,Porte de Saint-Ouen,48.897672,2.32983,True
1,"48.90101049575492, 2.2135566086401575","{""coordinates"": [2.213556608640157, 48.9010104...",578,388,Nanterre-Université,<NA>,<NA>,70956,Nanterre Université,43170,...,1,1,1972-10-01T01:00:00+01:00,642345.8425,6867065.7254,https://data.iledefrance-mobilites.fr/api/expl...,Nanterre-Université,48.90101,2.213557,True
2,"48.970665048322914, 2.0777317645655926","{""coordinates"": [2.077731764565593, 48.9706650...",6,127,Achères-Ville,<NA>,<NA>,73604,Achères Ville,46647,...,1,0,1988-05-29T02:00:00+02:00,632479.4115,6874918.7325,https://data.iledefrance-mobilites.fr/api/expl...,Achères-Ville,48.970665,2.077732,True
3,"48.88521121820875, 2.1559034718965178","{""coordinates"": [2.155903471896518, 48.8852112...",172,98,Châtou-Croissy,<NA>,<NA>,64483,Châtou - Croissy,53783,...,1,0,1972-10-01T01:00:00+01:00,638100.6669,6865352.6049,https://data.iledefrance-mobilites.fr/api/expl...,Châtou-Croissy,48.885211,2.155903,True
4,"48.83921929995196, 2.633210913417631","{""coordinates"": [2.633210913417631, 48.8392192...",473,6,Lognes,<NA>,<NA>,68123,Lognes,43152,...,1,0,1980-12-19T01:00:00+01:00,673078.4315,6859970.1835,https://data.iledefrance-mobilites.fr/api/expl...,Lognes,48.839219,2.633211,True


,data_type,missing_rows,distinct_values
column,,,
Geo Point,string,0,1183
Geo Shape,string,0,1183
gares_id,string,0,1287
id_gare_g,string,0,1001
nom_long,string,0,1018
nom_so_gar,string,1200,67
nom_su_gar,string,1258,10
id_ref_ZdC,string,0,999
nom_ZdC,string,0,1001


## Analyse modes and repeated identifiers

The cleaner parses `Geo Point` as latitude then longitude and flags coordinates outside
global bounds. Original `Geo Point`, `Geo Shape`, `x`, and `y` columns are retained.
Bounds do not establish the source CRS for every supplied coordinate field.


In [69]:
station_modes = stations.mode_summary(cleaned)
display(station_modes)
display(repeated_keys(cleaned, ['id_gare_g']).head(20))
display(cleaned.loc[~cleaned.coordinates_in_bounds])


,mode,station_line_records,distinct_station_ids
0,CABLE,5,5
1,METRO,409,325
2,RER,252,241
3,TRAIN,328,269
4,TRAMWAY,283,274
5,VAL,10,10


,id_gare_g,records
740,329,9
798,384,9
692,285,7
749,338,6
177,108,6
805,390,6
738,327,5
750,339,5
515,169,5
225,109069,5


,Geo Point,Geo Shape,gares_id,id_gare_g,nom_long,nom_so_gar,nom_su_gar,id_ref_ZdC,nom_ZdC,id_ref_ZdA,...,idf,principal,date_mise_,x,y,picto ligne,nom_iv,latitude,longitude,coordinates_in_bounds


In [70]:
stations.plot_modes(station_modes).show()
stations.plot_locations(cleaned).show()

## Play with one station and its listed transport lines

This helps identify stations with several listed modes or lines. Search by name first,
then select **one exact `id_gare_g`**: a search for Saint-Denis also finds different places
such as Saint-Denis Pleyel and Strasbourg-Saint-Denis. They must not be merged by name.

The chart shows one dot per distinct line listed at the selected station, grouped by mode.
For station `416` (Saint-Denis), inspect RER D, TRAIN H, TRAM 1 and TRAM 8 in this source.
The table provides the same information as text. Equal dots indicate presence, not service
frequency, passenger numbers or transfer quality.

**Coverage:** this railway-station source includes rail, metro, tram and other listed rail
modes. It does not list nearby bus connections. Listed lines do not prove current operation;
exact schedules, bus access and walking transfers require further data later.


In [74]:
SEARCH = 'Palaiseau'
candidates = cleaned.loc[cleaned.nom_long.str.contains(SEARCH, case=False, na=False, regex=False)]
display(candidates[['nom_long', 'id_gare_g']].drop_duplicates().sort_values('nom_long'))


,nom_long,id_gare_g
257,Massy – Palaiseau,169
483,Massy-Palaiseau,169
587,Palaiseau,172
1091,Palaiseau-Villebon,173


In [75]:
STATION_ID = '169'  # Choose an exact id_gare_g from the search results above.
selected = stations.select_station(cleaned, STATION_ID)

listed_lines = selected[['res_com', 'mode', 'idrefligc']].drop_duplicates()
print('Station:', selected.nom_long.iloc[0], '| ID:', STATION_ID)
print('Listed modes:', listed_lines['mode'].nunique(), '| Listed lines:', len(listed_lines))
display(listed_lines.sort_values(['mode', 'res_com']))
display(selected[['nom_long', 'id_gare_g', 'idrefligc', 'res_com', 'mode', 'latitude', 'longitude']])
stations.plot_station_lines(selected).show()


Station: Massy – Palaiseau | ID: 169
Listed modes: 4 | Listed lines: 5


,res_com,mode,idrefligc
257,METRO 18,METRO,C02833
1240,RER B,RER,C01743
483,RER C,RER,C01727
1230,TRAIN V,TRAIN,C02711
1013,TRAM 12,TRAMWAY,C02529


,nom_long,id_gare_g,idrefligc,res_com,mode,latitude,longitude
257,Massy – Palaiseau,169,C02833,METRO 18,METRO,48.725254,2.258586
483,Massy-Palaiseau,169,C01727,RER C,RER,48.725997,2.258323
1013,Massy-Palaiseau,169,C02529,TRAM 12,TRAMWAY,48.724729,2.257515
1230,Massy-Palaiseau,169,C02711,TRAIN V,TRAIN,48.725997,2.258323
1240,Massy-Palaiseau,169,C01743,RER B,RER,48.725098,2.259946


## Optional: save this dataset as Parquet



In [73]:
SAVE_CLEANED = False
output_path = ROOT / 'data' / 'cleaned' / 'stations.parquet'

if SAVE_CLEANED:
    save_parquet(cleaned, output_path)
    print('Saved', len(cleaned), 'rows to', output_path)
else:
    print('No cleaned data written. Change SAVE_CLEANED after reviewing this dataset.')


No cleaned data written. Change SAVE_CLEANED after reviewing this dataset.
